# 05_axis5
Axis 5 (longitudinal equity of recourse access).
Recourse is only fair if the prescribed change is comparably attainable across
groups. Using the empirically observed feasibility bounds from axis 4, we compute,
by sex / age band / income quintile / education / region, the share of at-risk
persons who realise a recourse-consistent BMI reduction, and track how that
attainment rate and between-group gap move across wave pairs. We also relate
recourse attainment to unmet-care experience as an external validity check.

In [1]:
%run 00_config.ipynb

PROJ_DIR: /home/claude/recourse_khp
1y pairs: [(2019, 2020), (2020, 2021), (2021, 2022), (2022, 2023), (2023, 2024)]
2y pairs: [(2019, 2021), (2020, 2022), (2021, 2023), (2022, 2024)]
registry loaded
helpers loaded
00_config ready


In [2]:
tr=pd.read_parquet(os.path.join(DATA_DIR,"transitions_1y.parquet"))
atr=tr[tr["HTN_atrisk"]==1].copy()
atr["red_BMI"]=-(atr["BMI_t1"]-atr["BMI_t0"])
# recourse-consistent attainment: realise a reduction >= population 75th pct bound
tau=float(atr["red_BMI"][atr["red_BMI"]>0].quantile(0.75))
atr["attain"]=(atr["red_BMI"]>=tau).astype(int)
print(f"attainment threshold tau = {tau:.2f} BMI units")
print("overall attainment rate:",round(atr["attain"].mean(),3))

attainment threshold tau = 1.28 BMI units
overall attainment rate: 0.076


In [3]:
# --- Attainment rate by group (pooled over wave pairs) ---
def rate_by(df, col):
    g=df.dropna(subset=[col]).groupby(col, observed=True)["attain"]
    out=g.agg(["mean","size"]).reset_index().rename(
        columns={col:"level","mean":"attain_rate","size":"n"})
    out.insert(0,"stratum",col); out["attain_rate"]=out["attain_rate"].round(3)
    return out
strata=["SEX_t0","AGEG_t0","INCQ_t0","EDU_t0","REGION1_t0"]
strata=[s for s in strata if s in atr.columns]
eq=pd.concat([rate_by(atr,s) for s in strata], ignore_index=True)
savetable(eq,"t05_attainment_by_group", index=False)
print(eq.to_string(index=False))

saved: t05_attainment_by_group.csv
   stratum level  attain_rate     n
    SEX_t0     F        0.078 17195
    SEX_t0     M        0.074 13658
   AGEG_t0 19-29        0.089  3071
   AGEG_t0 30-39        0.099  4106
   AGEG_t0 40-49        0.071  5930
   AGEG_t0 50-59        0.069  5884
   AGEG_t0 60-69        0.063  6412
   AGEG_t0   70+        0.080  5450
   INCQ_t0    Q1        0.086  5124
   INCQ_t0    Q2        0.076  6074
   INCQ_t0    Q3        0.072  6578
   INCQ_t0    Q4        0.077  6415
   INCQ_t0    Q5        0.071  6522
    EDU_t0   1.0        0.123   636
    EDU_t0   2.0        0.086  3852
    EDU_t0   3.0        0.069  3358
    EDU_t0   4.0        0.070  9628
    EDU_t0   5.0        0.077 12145
    EDU_t0   6.0        0.075  1234
REGION1_t0  11.0        0.076  2797
REGION1_t0  26.0        0.081  2222
REGION1_t0  27.0        0.064  1147
REGION1_t0  28.0        0.080  1812
REGION1_t0  29.0        0.086  1844
REGION1_t0  30.0        0.063  1284
REGION1_t0  31.0        0.070

In [4]:
# --- Between-group gap over wave pairs (age bands: youngest vs oldest) ---
def gap_over_time(df, col, hi, lo):
    rows=[]
    for pr,g in df.groupby("pair", observed=True):
        gg=g.dropna(subset=[col])
        a=gg.loc[gg[col]==hi,"attain"].mean()
        b=gg.loc[gg[col]==lo,"attain"].mean()
        rows.append({"pair":pr,f"{hi}":round(a,3),f"{lo}":round(b,3),
                     "gap":round((a-b),3) if pd.notna(a) and pd.notna(b) else np.nan})
    return pd.DataFrame(rows)
gap_age=gap_over_time(atr,"AGEG_t0","19-29","70+")
savetable(gap_age,"t05_gap_age_over_time", index=False)
print(gap_age.to_string(index=False))

saved: t05_gap_age_over_time.csv
     pair  19-29   70+    gap
2019_2020  0.137 0.150 -0.013
2020_2021  0.074 0.054  0.020
2021_2022  0.079 0.056  0.023
2022_2023  0.079 0.059  0.020
2023_2024  0.069 0.077 -0.008


In [5]:
# --- Axis 5 figure: attainment rate by age band across wave pairs ---
order=["19-29","30-39","40-49","50-59","60-69","70+"]
piv=(atr.dropna(subset=["AGEG_t0"])
        .groupby(["pair","AGEG_t0"], observed=True)["attain"].mean().reset_index())
piv=piv[piv["AGEG_t0"].isin(order)]
fig,ax=plt.subplots(figsize=(6.6,4.2))
greys=["#111111","#333333","#555555","#777777","#9a9a9a","#bcbcbc"]
for ag,gc in zip(order,greys):
    s=piv[piv["AGEG_t0"]==ag]
    ax.plot(s["pair"], s["attain"], marker="o", ms=4, color=gc, lw=1.2, label=ag)
ax.set_xlabel("Wave pair"); ax.set_ylabel("Recourse-consistent attainment rate")
ax.tick_params(axis="x", rotation=30)
ax.legend(frameon=False, fontsize=8, ncol=2, title=None)
savefig(fig,"f05_attainment_by_age"); plt.close(fig)
print("axis-5 figure saved")

saved: f05_attainment_by_age.png / f05_attainment_by_age.pdf
axis-5 figure saved


In [6]:
# --- Axis 5 figure: attainment by income quintile (pooled) ---
if "INCQ_t0" in atr.columns:
    iq=(atr.dropna(subset=["INCQ_t0"]).groupby("INCQ_t0", observed=True)["attain"]
        .mean().reindex(["Q1","Q2","Q3","Q4","Q5"]))
    fig,ax=plt.subplots(figsize=(5.2,4.0))
    ax.bar(iq.index.astype(str), iq.values, color="#888888", edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Household income quintile")
    ax.set_ylabel("Recourse-consistent attainment rate")
    savefig(fig,"f05_attainment_by_income"); plt.close(fig)
    print("income figure saved:", iq.round(3).to_dict())

saved: f05_attainment_by_income.png / f05_attainment_by_income.pdf
income figure saved: {'Q1': 0.086, 'Q2': 0.076, 'Q3': 0.072, 'Q4': 0.077, 'Q5': 0.071}


In [7]:
# --- External validity: does higher attainment track lower unmet-care? ---
if "UNMET_t1" in atr.columns:
    sub=atr.dropna(subset=["UNMET_t1"])
    tab=sub.groupby("attain")["UNMET_t1"].mean().rename("unmet_rate").reset_index()
    from scipy.stats import chi2_contingency
    ct=pd.crosstab(sub["attain"], sub["UNMET_t1"])
    chi2,p,_,_=chi2_contingency(ct)
    tab["chi2_p"]=round(p,4)
    savetable(tab,"t05_attain_vs_unmet", index=False)
    print(tab.to_string(index=False))

saved: t05_attain_vs_unmet.csv
 attain  unmet_rate  chi2_p
      0    0.119572  0.1136
      1    0.131133  0.1136


In [8]:
# --- Summary panel: consolidate headline numbers across all axes ---
summary={
 "axis1_median_prescribed_BMI_reduction": None,
 "axis1_pop_achievement_at_median": None,
 "axis2_treatment_OR": None,
 "axis2_ipw_risk_diff_pp": None,
 "axis5_overall_attainment": round(float(atr["attain"].mean()),3),
 "axis5_attainment_gap_young_minus_old_pooled": None,
}
try:
    a1=pd.read_csv(os.path.join(TAB_DIR,"t03_axis1_achievement.csv"))
    summary["axis1_median_prescribed_BMI_reduction"]=float(a1.iloc[0]["prescribed_reduction"])
    summary["axis1_pop_achievement_at_median"]=float(a1.iloc[0]["pop_achievement_rate"])
except Exception as ex: print("a1",ex)
try:
    a2=pd.read_csv(os.path.join(TAB_DIR,"t04_axis2_ipw_logit.csv"))
    summary["axis2_treatment_OR"]=float(a2.loc[a2["term"]=="TREAT","OR"].iloc[0])
    rd=pd.read_csv(os.path.join(TAB_DIR,"t04_axis2_ipw_riskdiff.csv"))
    summary["axis2_ipw_risk_diff_pp"]=round(float(rd.iloc[0]["risk_diff"])*100,3)
except Exception as ex: print("a2",ex)
gp=gap_age["gap"].mean()
summary["axis5_attainment_gap_young_minus_old_pooled"]=round(float(gp),3)
S=pd.DataFrame([summary]).T.rename(columns={0:"value"})
savetable(S,"t05_headline_summary")
print(S.to_string())

saved: t05_headline_summary.csv
                                              value
axis1_median_prescribed_BMI_reduction        4.3400
axis1_pop_achievement_at_median              0.0050
axis2_treatment_OR                           0.6084
axis2_ipw_risk_diff_pp                      -1.0090
axis5_overall_attainment                     0.0760
axis5_attainment_gap_young_minus_old_pooled  0.0080
